# 12 · Web / API / 文件工具

用 Python 與網路世界溝通，並用程式自動生成與批次產出文件。

## 學習目標

- 能用 requests 函式庫送出 HTTP（hypertext transfer protocol）請求，理解 get 與 post 的差異、判讀狀態碼（status code）、解析 json 回應。
- 理解 REST API（representational state transfer）的基本互動模式，以及金鑰（API key）在驗證與安全上的角色與保管原則。
- 熟悉資料編碼概念，能用 base64 將二進位資料轉成可放進文字傳輸的字串。
- 能用 urllib.parse 組裝與拆解網址，並用 mimetypes 判斷檔案的內容型別（MIME type）。
- 能用 subprocess 從 Python 呼叫外部命令列工具，取得其輸出。
- 能用字串模板生成 HTML 文件，理解 markdown 轉 HTML 的概念，並建立批次自動產出文件的設計思維。

## HTTP 請求基礎與 requests

HTTP（hypertext transfer protocol）是程式與網路服務溝通的共同語言：你送出一個請求（request），對方回一個回應（response）。

為什麼先學它：所有網路操作的根基都是「送出請求、判讀是否成功、再取出資料」這三步。

兩種常見方法的差異：

| 方法 | 用途 | 參數位置 |
|---|---|---|
| get | 取資料（查詢） | 放在網址查詢字串 |
| post | 送資料（建立 / 提交） | 放在請求主體 body |

回應裡常用的兩個出口：`text`（原始文字）與 `json()`（把 json 文字解析成 Python 物件）。判讀成功與否則看狀態碼（status code）：2xx 成功、4xx 用戶端錯誤（如 404 找不到）、5xx 伺服器錯誤（如 500）。

In [ ]:
# 概念：用自造的模擬回應物件示範「送請求 -> 檢查狀態碼 -> 取資料」三步
# 不連真實網路，改用一個假的 client，回傳行為與 requests 相同的物件

class FakeResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code   # 狀態碼：判斷成功（200）或失敗
        self._payload = payload          # 內部存放要回傳的資料字典

    @property
    def text(self):
        import json as _json
        return _json.dumps(self._payload, ensure_ascii=False)   # text：未解析的原始字串

    def json(self):
        return self._payload   # json()：把回應解析成 Python 物件（這裡直接給字典）

def fake_get(url):
    # 造假：依網址決定回傳。已知路徑給 200，其餘給 404
    if url.endswith('/buildings'):
        return FakeResponse(200, {'count': 2, 'items': ['塔樓A', '塔樓B']})
    return FakeResponse(404, {'error': 'not found'})

resp = fake_get('https://example.local/buildings')   # 送出 get 請求
print('狀態碼:', resp.status_code)

# 注意：先確認狀態碼成功，再呼叫 json()，避免對錯誤回應解析出垃圾
if resp.status_code == 200:
    data = resp.json()
    print('資料筆數:', data['count'])
    print('項目:', data['items'])
else:
    print('請求失敗，不解析資料')

## REST API 與金鑰概念

REST API（representational state transfer）是多數對外服務提供資料的風格：每個資源有一個端點（endpoint，一個網址），你帶上查詢參數（query parameter）與請求標頭（header）去問，服務回對應結果。

金鑰（API key）是用來驗證（authentication）你是誰的憑證，常放在請求標頭裡。

為什麼要學保管原則：金鑰一旦寫死在程式碼或外洩，任何人都能冒用你的額度與權限。正確做法是從環境變數或設定檔讀取，不硬編（hard-code）在原始碼。

形狀：

```
端點 + 查詢參數 -> 服務 -> 回應
標頭 {"Authorization": "Bearer <金鑰>"}
```

In [ ]:
# 概念：自造本機假 API，示範查詢參數與金鑰標頭如何影響回傳結果
import os

VALID_KEY = 'demo-key-123'   # 僅供示範；真實情境金鑰絕不寫死在程式碼

def fake_api(endpoint, params, headers):
    # 先驗證金鑰：標頭沒有正確 key 就回 401（未授權）
    if headers.get('X-API-Key') != VALID_KEY:
        return 401, {'error': 'unauthorized'}
    # 依查詢參數回傳不同結果：這裡用 min_floors 過濾假資料
    buildings = [('塔樓A', 25), ('街屋B', 4), ('商辦C', 18)]
    min_floors = params.get('min_floors', 0)
    hits = [name for name, floors in buildings if floors >= min_floors]
    return 200, {'endpoint': endpoint, 'result': hits}

# 技巧：金鑰從環境變數取，取不到才用示範預設，避免硬編到正式碼
api_key = os.environ.get('BUILDING_API_KEY', VALID_KEY)

status, body = fake_api('/buildings', {'min_floors': 10}, {'X-API-Key': api_key})
print('帶正確金鑰:', status, body)

status2, body2 = fake_api('/buildings', {'min_floors': 10}, {'X-API-Key': 'wrong'})
print('帶錯誤金鑰:', status2, body2)

## base64 與資料編碼

很多協定（例如電子郵件、json 欄位）只能傳純文字，但圖片、檔案等是二進位（binary）資料。編碼（encoding）就是換一種表示法，把位元組（bytes）轉成安全可傳輸的文字。

base64 是最常見的一種：用 64 個可列印字元表示任意位元組。

重點釐清：編碼不等於加密（encryption）。base64 只是換表示法，任何人都能還原，沒有保密效果；要保密得另外加密。

形狀：

```
bytes --base64 編碼--> 文字字串 --base64 解碼--> 原 bytes
```

In [ ]:
# 概念：把二進位資料做 base64 編碼再還原，驗證前後一致
import base64

# 造一段假的二進位資料當示範（中文字串先以 utf-8 轉成 bytes）
raw_bytes = '建築模型#42'.encode('utf-8')
print('原始 bytes:', raw_bytes)

encoded = base64.b64encode(raw_bytes)        # 編碼：bytes -> base64 的 bytes
encoded_text = encoded.decode('ascii')       # 轉成純文字字串，方便放進文字傳輸
print('base64 文字:', encoded_text)

decoded = base64.b64decode(encoded_text)     # 解碼：還原回原始 bytes
print('還原 bytes:', decoded)
print('前後一致:', decoded == raw_bytes)

# 注意：base64 可被任何人還原，不具保密性，不能拿來當加密用
print('看得懂的明碼:', decoded.decode('utf-8'))

## 網址解析與 MIME 型別

網址（URL）有固定結構（協定、主機、路徑、查詢字串）。手動拼字串容易在特殊字元（中文、空白、`&`）出錯，所以用 urllib.parse 來正確組裝與拆解。

- `urlencode`：把參數字典編成查詢字串，並自動做百分號編碼（percent-encoding）。
- `urlparse`：把一條網址拆成各部位。

MIME 型別（MIME type，例如 `text/csv`、`image/png`）告訴程式一份資料該怎麼處理，是上傳下載與生成文件的共同語彙。`mimetypes.guess_type` 可依副檔名猜測型別。

形狀：

```
基底網址 + "?" + urlencode(參數字典)
```

In [ ]:
# 概念：用 urlencode 組查詢字串、urlparse 拆網址、mimetypes 猜型別
from urllib.parse import urlencode, urlparse, parse_qs
import mimetypes

# 造一筆查詢條件（含中文，需百分號編碼才安全）
params = {'district': '中正區', 'min_floors': 10}
query = urlencode(params)                          # 自動把中文與符號做 percent-encoding
base = 'https://example.local/buildings'
url = base + '?' + query
print('組好的網址:', url)

parsed = urlparse(url)                             # 拆解：取出主機、路徑、查詢字串等
print('主機:', parsed.netloc, '| 路徑:', parsed.path)
print('拆回字典:', parse_qs(parsed.query))         # 查詢字串再還原成字典

# 依副檔名猜內容型別：下載 csv 檔時告訴瀏覽器這是表格資料
mime, _ = mimetypes.guess_type('buildings.csv')
print('buildings.csv 的 MIME 型別:', mime)

## 用 subprocess 呼叫外部工具

Python 不必重造輪子，可以直接驅動既有的命令列工具（command-line tool）並接收它的輸出。這是串接系統工具的關鍵能力。

核心是 `subprocess.run`，把命令與參數寫成一個列表（list）傳入，回傳物件包含：

- 回傳碼（return code）：0 代表成功，非 0 代表失敗。
- 標準輸出（stdout）：工具印出的正常結果。
- 標準錯誤（stderr）：工具印出的錯誤訊息。

形狀：

```
subprocess.run(["命令", "參數1", "參數2"], capture_output=True, text=True)
```

In [ ]:
# 概念：用 subprocess 呼叫外部工具，擷取輸出並檢查回傳碼
import subprocess
import sys

# 技巧：跨平台示範就直接呼叫目前的 Python 直譯器跑一小段程式
# 命令與參數用 list 傳入，避免 shell 特殊字元解讀問題
cmd = [sys.executable, '-c', "print('來自外部程序的輸出')"]

result = subprocess.run(cmd, capture_output=True, text=True)   # capture_output 擷取 stdout/stderr，text 以文字解碼

print('回傳碼:', result.returncode)        # 0 表示成功
print('標準輸出:', result.stdout.strip())
print('標準錯誤:', repr(result.stderr))

# 注意：用回傳碼判斷成功與否，而非只看有沒有輸出
print('執行成功:', result.returncode == 0)

## 用程式生成文件

把資料與版型分離：先寫好一份含佔位符（placeholder）的模板（template），再用變數把內容填進去，就能大量產生結構一致的文件。

為什麼這樣設計：版型只寫一次，資料換一批就重新產出一份，維護成本低。

markdown 與 HTML 的關係：markdown 是給人寫的精簡標記，最終常被轉成 HTML 給瀏覽器顯示；理解兩者對應有助於選擇輸出格式。

安全意識：若資料來自外部，填入 HTML 前要做跳脫（escaping），把 `<`、`>`、`&` 換成安全寫法，避免破壞版面或被注入。

In [ ]:
# 概念：用含佔位符的 HTML 模板字串，填入資料字典產出完整 HTML
from html import escape

# 造一筆假資料當示範
building = {'name': '中正塔 <示範>', 'floors': 25, 'height': 88.5}

# 模板用 {名稱} 當佔位符，之後用 format 填值
template = (
    '<div class="card">\n'
    '  <h2>{name}</h2>\n'
    '  <p>樓層數：{floors}</p>\n'
    '  <p>樓高：{height} 公尺</p>\n'
    '</div>'
)

# 注意：資料含 < > 等字元，填入前先 escape，避免破壞 HTML 結構
html = template.format(
    name=escape(str(building['name'])),
    floors=building['floors'],
    height=building['height'],
)
print(html)

## 自動化批次產出的設計思維

單筆能跑只是起點。把流程拆成可重複、可容錯、可批量的步驟，才是自動化的核心。

幾個關鍵概念：

- 批次處理（batch processing）：對一份資料清單逐筆套用相同流程。
- 失敗處理與重試（retry）：遇到壞掉的一筆，記錄並跳過或重試，不讓整批中斷。
- 可重複執行（idempotency）：同樣輸入重跑多次，結果一致、不重複產生副作用。

設計原則：把「輸入資料 -> 套模板 -> 輸出文件」做成一個可逐筆呼叫的函式，外層用迴圈批次驅動，並收集成功與失敗清單。

In [ ]:
# 概念：批次套模板產文件，遇到問題項目記錄並跳過而不中斷整批
template = '<li>{name}：{floors} 層</li>'

# 造一份多筆資料，刻意混入一筆缺欄位的壞資料
records = [
    {'name': '塔樓A', 'floors': 25},
    {'name': '街屋B'},                 # 缺 floors，會在套模板時出錯
    {'name': '商辦C', 'floors': 18},
]

outputs = []
skipped = []   # 容錯：把失敗項目記下來，整批跑完再回報
for i, rec in enumerate(records):
    try:
        outputs.append(template.format(**rec))   # 缺鍵會丟 KeyError
    except KeyError as e:
        skipped.append((i, str(e)))               # 記錄哪一筆、缺什麼，然後繼續

print('成功產出:')
for line in outputs:
    print(' ', line)
print('被跳過的項目（索引, 原因）:', skipped)

## 練習

以下三題由淺入深，皆為整合多個概念的小任務。資料請自己用 list 造，不引用任何外部檔案。只需在 `# TODO` 處完成。

In [ ]:
# TODO 1 ·（簡單）建築清單查詢網址（整合：網址解析 urlencode/urlparse + MIME 型別）
#   情境：替一份建築資料下載頁組出帶查詢參數的網址，並標出下載檔案的型別。
#   要求：
#     1. 在程式內用字典自造一筆查詢條件，例如 {"district": "中正區", "min_floors": 10}，
#        用 urllib.parse.urlencode 編成查詢字串，再接上一個基底網址。
#     2. 給定下載檔名 "buildings.csv"，用 mimetypes.guess_type 查出 MIME 型別並印出。
#   驗收：應看到一條把中文與數字正確百分號編碼後的完整網址，以及 text/csv 這個 MIME 型別。

# TODO: 在這裡完成

In [ ]:
# TODO 2 ·（中間）建築資料抓取報表（整合：HTTP 請求 + 狀態碼 + json 解析 + HTML 生成）
#   情境：從一個自造的本機假 API 取回多筆建築資料，整理成一份 HTML 報表。
#   要求：
#     1. 自寫一個假 API 函式，回傳含 status_code 與 json() 方法的模擬回應物件，
#        內含數筆建築資料（每筆有 height、floors、footprint，用 list 自造仿真數字）。
#     2. 呼叫後先檢查狀態碼是否為 200，成功才呼叫 json() 取資料；非 200 則印錯誤並停止。
#     3. 為每筆算出樓地板面積 GFA（gross floor area，以 footprint × floors 估算），
#        把所有欄位填進一份含佔位符的 HTML 模板，每筆成為表格一列。
#     4. 彙整成單一 HTML 報表字串並印出。
#   驗收：應看到一段完整 HTML，表格中每棟建築一列，且含一欄由程式算出的 GFA。

# TODO: 在這裡完成

In [ ]:
# TODO 3 ·（稍難）容積政策情境的街廓聚合比較
#   （整合：json 資料 + numpy 聚合 + 批次產出 + 文件生成 + 容錯）
#   情境：給定多個街廓 block 內的建築資料，比較容積率 FAR 政策調整前後各街廓總樓地板面積變化，
#         並輸出一份摘要文件。
#   要求：
#     1. 用 list 或 numpy 自造數筆建築資料，每筆含街廓編號 cell、footprint、floors，
#        其中刻意混入一兩筆缺漏值（例如 floors 為 None）。
#     2. 設計政策情境：對符合條件的街廓套用高度乘數 multiplier（例如允許樓層數 ×1.5），
#        算出調整後 GFA；遇缺漏項目要跳過或記錄而不中斷整批（容錯）。
#     3. 依街廓編號 cell 聚合出政策前、政策後的總 GFA，並算出每個街廓的增幅百分比。
#     4. 把各街廓的前後對照與增幅，用模板批次填成 HTML 或 markdown 摘要，
#        並在末尾列出被跳過的缺漏項目。
#   驗收：應看到一份依街廓分組的對照摘要，含政策前後總 GFA 與增幅百分比，
#         且明確標示哪些缺漏項目被略過。

# TODO: 在這裡完成

## 小結

- HTTP 操作的根基是三步：送出請求、用狀態碼（status code）判讀是否成功、再用 `json()` 取出結構化資料。
- REST API 以端點加查詢參數組成查詢，金鑰（API key）用於驗證且絕不可硬編或外洩，應從環境變數或設定取得。
- base64 是編碼（換表示法）而非加密，任何人都能還原，僅用於把二進位資料塞進文字通道。
- urllib.parse 負責安全組裝與拆解網址（含百分號編碼），mimetypes 依副檔名判斷內容型別（MIME type）。
- subprocess.run 讓 Python 驅動外部命令列工具，用回傳碼（return code）判斷成敗，並擷取 stdout 與 stderr。
- 文件生成的核心是資料與模板分離，再以批次處理（batch processing）配合容錯，把單筆流程擴展成可重複、可容錯的自動化。